# ML06 · PyTorch 張量與 autograd

用 PyTorch 的張量（tensor）與自動微分（autograd），讓框架替你算梯度，並親手把梯度下降（gradient descent）重做一遍。

## 學習目標
- 能建立並操作 PyTorch 張量（tensor），理解它與 numpy 陣列的關係與差異。
- 理解 requires_grad 如何開啟對張量的梯度追蹤，並建立計算圖（computation graph）的直覺。
- 能呼叫 backward 觸發反向傳播（backpropagation），並從 .grad 讀出梯度（gradient）。
- 看懂框架自動算出的梯度，與手寫公式得到的結果一致。
- 用 autograd 重寫一次梯度下降（gradient descent），體會「不用手推導數」的便利。

In [ ]:
# 概念：載入 PyTorch 並固定亂數種子，讓每次執行結果一致（方便對照）
import torch

torch.manual_seed(0)   # 固定種子，後面造的假資料與隨機初始化才可重現

print("PyTorch 版本:", torch.__version__)

## 張量基礎：建立與運算

張量（tensor）是 PyTorch 的基本資料容器，可視為「會記得運算過程」的 numpy 陣列。先把建立與運算的手感建立起來，後面的自動微分才有對象可操作。

為什麼重要：深度學習的所有輸入、參數、輸出都是張量；它和 numpy 陣列一樣有形狀（shape）與資料型別（dtype），但能在 GPU 上運算，並支援自動微分。

常見建立方式：`torch.tensor(資料)`、`torch.zeros(形狀)`、`torch.arange(起, 迄, 步長)`。

In [ ]:
# 概念：用多種方式建立張量，並檢查形狀 shape 與資料型別 dtype
import torch

# 造一份假建築資料：五棟樓的樓高（公尺）與面積（平方公尺）
heights = torch.tensor([12.0, 30.0, 9.0, 45.0, 18.0])      # 用 list 直接建成一維張量
areas = torch.tensor([85.0, 120.0, 60.0, 200.0, 95.0])

zeros = torch.zeros(5)              # 長度 5、全為 0 的張量，常用來初始化
ramp = torch.arange(0, 5, 1)        # 0,1,2,3,4；類似 numpy.arange

print("樓高:", heights)
print("面積:", areas)
print("zeros:", zeros, " arange:", ramp)
# shape 讀作各軸長度；dtype 預設為 float32（list 含小數時）
print("樓高 shape:", heights.shape, " dtype:", heights.dtype)
# 注意：arange 給整數時 dtype 會是 int64，與 float 張量運算前常需轉型
print("arange dtype:", ramp.dtype)

### 張量運算與 numpy 互轉

張量支援逐元素（element-wise）的加減乘除與向量化運算，寫法與 numpy 幾乎相同；矩陣乘法（matrix multiplication）用 `@` 或 `torch.matmul`。

張量與 numpy 陣列可互轉：`tensor.numpy()` 轉出、`torch.from_numpy(陣列)` 轉入。何時用：常在資料準備階段用 numpy，餵進模型前轉成張量。

In [ ]:
# 概念：張量的逐元素運算、矩陣乘法，以及與 numpy 的互轉
import torch
import numpy as np

heights = torch.tensor([12.0, 30.0, 9.0, 45.0, 18.0])
areas = torch.tensor([85.0, 120.0, 60.0, 200.0, 95.0])

# 逐元素：每棟樓高加 3 公尺（向量化，不需迴圈）
print("樓高各加 3:", heights + 3)
# 逐元素相乘：可粗略當成「樓高乘面積」的量體指標
print("樓高×面積:", heights * areas)

# 矩陣乘法：把兩個一維向量做內積，得到一個純量
dot = heights @ areas                 # @ 即 matmul；一維對一維是內積
print("內積（純量）:", dot)

# 互轉：張量 -> numpy -> 張量
as_numpy = heights.numpy()            # 轉成 numpy 陣列
back = torch.from_numpy(as_numpy)     # 再轉回張量
print("轉回張量是否一致:", torch.equal(heights, back))

## requires_grad 與計算圖

把張量的 `requires_grad` 設為 True，PyTorch 就會在背後記錄對它做的每一步運算，串成一張可回溯的計算圖（computation graph）。這是自動微分的前提。

幾個關鍵名詞：
- 葉節點（leaf node）：我們自己建立、要對它求梯度的張量（例如參數 w、b）。
- grad_fn：由運算自動產生的張量會帶一個 grad_fn，記錄「它是由哪種運算算出來的」，反向時靠它回推。

形狀（示意）：`w = torch.tensor(值, requires_grad=True)`。

In [ ]:
# 概念：開啟 requires_grad 後，運算結果會帶 grad_fn，記錄計算圖
import torch

# 把 w、b 當成可訓練參數：requires_grad=True 才會被追蹤
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
x = torch.tensor(3.0)                 # 輸入資料不需梯度，維持預設 False

y = w * x + b                         # 一個簡單純量運算，會被記進計算圖

print("y =", y)
# 葉節點 w、b 的 is_leaf 為 True；y 是運算結果，不是葉節點
print("w 是葉節點:", w.is_leaf, " y 是葉節點:", y.is_leaf)
# grad_fn 記錄 y 由哪種運算產生（這裡是加法 AddBackward），反向時靠它回推
print("y.grad_fn:", y.grad_fn)
# 注意：未開啟 requires_grad 的 x 沒有 grad_fn，不會被追蹤
print("x.requires_grad:", x.requires_grad)

## backward 與讀取 .grad

自動微分（autograd）的核心動作是 `backward`：它從輸出沿計算圖往回走，把每個葉節點的梯度（gradient）算出來，填進該張量的 `.grad`。

兩個務必記住的細節：
- 梯度累積（gradient accumulation）：`.grad` 是「累加」的，每輪訓練前要用 zero_grad 清零，否則梯度會疊加導致更新錯誤。
- no_grad 區塊：只想前向推論、不想被追蹤時，用 `with torch.no_grad():` 暫停建圖，省記憶體也避免誤算梯度。

In [ ]:
# 概念：呼叫 backward 觸發反向傳播，從 .grad 讀梯度，並示範清零與累積
import torch

w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
x = torch.tensor(3.0)

y = w * x + b
y.backward()                          # 從 y 往回走計算圖，把梯度填進 w.grad、b.grad

# dy/dw = x = 3、dy/db = 1，可手算驗證
print("第一次 w.grad:", w.grad, " b.grad:", b.grad)

# 注意：不清零再算第二次，.grad 會累加（變成兩次的和）
y2 = w * x + b
y2.backward()
print("未清零的第二次 w.grad:", w.grad)   # 變成 3+3=6，這通常不是我們要的

# 技巧：每輪反向前先清零，梯度才正確
w.grad.zero_()                        # 就地清零，底線結尾代表 in-place
b.grad.zero_()
y3 = w * x + b
y3.backward()
print("清零後重算 w.grad:", w.grad, " b.grad:", b.grad)

# 概念：no_grad 區塊內的運算不建計算圖、不追蹤梯度
with torch.no_grad():
    preview = w * x + b               # 只想看結果、不需要梯度
print("no_grad 下的結果 requires_grad:", preview.requires_grad)

## 梯度的直覺：框架在算什麼

梯度（gradient）告訴你「往哪個方向、調多少，能讓損失（loss）下降」。對參數而言，梯度是損失對該參數的偏導數（partial derivative）。

關鍵極簡公式：對平方誤差損失 loss = (y_pred − y)^2，梯度方向由 2 (y_pred − y) 主導。
- 梯度為正：表示參數增大會讓損失上升，所以應往減小的方向調。
- 梯度為負：表示參數增大會讓損失下降，應往增大的方向調。

下面用一維線性關係比對：手算公式與 autograd 應得到相同的梯度。

In [ ]:
# 概念：對平方誤差損失，比對手算梯度公式與 autograd 結果是否一致
import torch

# 自造一筆一維線性資料：用單一斜率 w 預測 y_pred = w * x
x = torch.tensor(4.0)
y_true = torch.tensor(10.0)           # 真實答案
w = torch.tensor(2.0, requires_grad=True)

y_pred = w * x
loss = (y_pred - y_true) ** 2         # 平方誤差 squared error
loss.backward()                       # autograd 算出 dloss/dw

# 手算：dloss/dw = 2 (w*x - y) * x（鏈鎖律），用同樣的數值代入
manual_grad = 2 * (w.item() * x.item() - y_true.item()) * x.item()

print("autograd 梯度:", w.grad.item())
print("手算公式梯度:", manual_grad)
# 兩者應幾乎相同；浮點誤差用 isclose 判斷
print("兩者是否一致:", torch.isclose(w.grad, torch.tensor(manual_grad)).item())
# 梯度為正代表 w 增大會讓損失上升，故應把 w 往下調
print("梯度為正嗎（正=該調小 w）:", w.grad.item() > 0)

## 用 autograd 重做梯度下降

梯度下降（gradient descent）的步驟不變：算損失 → 算梯度 → 沿梯度反方向更新參數。差別只在於「算梯度」這一步改用 backward 讀 .grad，不必再手推導數。

更新公式：`參數 = 參數 − 學習率 × 梯度`。學習率（learning rate）控制每步走多大：太小收斂慢、太大可能發散。

迴圈要點：更新參數時包在 no_grad 內（更新本身不該被追蹤），更新後記得 zero_grad，否則梯度會累加。

In [ ]:
# 概念：用 autograd 跑完整梯度下降，擬合一條帶雜訊的線性資料
import torch

torch.manual_seed(0)

# 自造資料：真實關係 y = 3x + 2，加一點高斯雜訊讓它更像真實量測
x = torch.linspace(0, 5, 30)                       # 0 到 5 取 30 個等距點
y = 3.0 * x + 2.0 + 0.5 * torch.randn(30)          # 真實斜率 3、截距 2

# 兩個可訓練參數，從隨意初始值開始學
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
learning_rate = 0.02                               # 學習率：每步更新的幅度

for step in range(200):
    y_pred = w * x + b                             # 前向：線性預測
    loss = ((y_pred - y) ** 2).mean()              # 均方誤差 mean squared error
    loss.backward()                                # 反向：算 w.grad、b.grad

    # 注意：更新參數時不該被追蹤，否則更新動作也會進計算圖
    with torch.no_grad():
        w -= learning_rate * w.grad                # 沿梯度反方向走一步
        b -= learning_rate * b.grad
        w.grad.zero_()                             # 清零，避免下一輪梯度累加
        b.grad.zero_()

    if step % 40 == 0:                             # 每 40 步印一次損失觀察下降
        print(f"step {step:3d}  loss={loss.item():.4f}  w={w.item():.3f}  b={b.item():.3f}")

# 最終參數應接近真實的 w=3、b=2
print("最終 w:", round(w.item(), 3), " 最終 b:", round(b.item(), 3))

## 練習

以下三題由淺入深，皆為複合型但簡短。資料請自己用 list 或 torch 造（仿真即可），完成後對照「驗收」確認輸出。

In [ ]:
# TODO 1 ·（簡單）樓高張量小計算（整合：張量建立 + 張量運算）
#   情境：用 list 自造五棟樓的樓高（公尺）與樓層數，想算每棟的平均層高。
#   要求：
#     1. 把樓高與樓層數各建成一個張量，檢查形狀 shape 與 dtype。
#     2. 用張量運算逐棟相除得到平均層高，並印出整體平均（用 .mean()）。
#   驗收：應該看到一個長度為五的平均層高張量，與一個整體平均純量。
# TODO: 在這裡完成


In [ ]:
# TODO 2 ·（中間）一個參數的梯度檢查（整合：requires_grad + backward + 讀 .grad + 梯度直覺）
#   情境：自造一組「容積率（floor area ratio）對土地價格」的線性資料（含少量雜訊），
#         先用單一斜率參數 w 做預測 y_pred = w * x。
#   要求：
#     1. 把 w 設為 requires_grad=True，用平方誤差（squared error）算出損失。
#     2. 呼叫 backward，讀出 w.grad。
#     3. 另用手寫公式 2*(w*x - y)*x（對所有樣本取平均）算同一個梯度，印出兩者比對。
#   驗收：應看到 autograd 與手算梯度數值幾乎相同，
#         並能說出梯度正負代表 w 該往下或往上調。
# TODO: 在這裡完成


In [ ]:
# TODO 3 ·（稍難）用 autograd 擬合樓高與造價（整合：張量 + 計算圖 + backward + zero_grad + 梯度下降）
#   情境：自造一批樓高對造價的資料（線性趨勢加雜訊），目標用 w 與 b 兩個參數擬合一條直線。
#   要求：
#     1. 設定 w、b 為可訓練張量（requires_grad=True），寫出預測與均方誤差損失。
#     2. 跑梯度下降迴圈：每輪 backward、讀 .grad、在 no_grad 下更新參數，更新後記得 zero_grad。
#     3. 記錄損失隨迭代變化，並比較兩種學習率（例如 0.01 與一個明顯過大的值）對收斂的影響。
#   驗收：應看到損失逐步下降、擬合出的 w 與 b 接近自造資料的真實趨勢，
#         且過大學習率會使損失發散（變大甚至變成 nan）。
# TODO: 在這裡完成


## 小結

- 張量（tensor）是 PyTorch 的基本資料容器，與 numpy 陣列同源、可互轉，同樣有 shape 與 dtype，但能在背後記錄運算以支援自動微分。
- 把張量的 requires_grad 設為 True，PyTorch 會把對它的每步運算串成計算圖（computation graph）；自己建立的參數是葉節點（leaf node），運算結果帶有 grad_fn。
- backward 從輸出沿計算圖往回走，把梯度填進葉節點的 .grad；.grad 會累積，每輪務必 zero_grad，只想推論時用 no_grad 暫停建圖。
- 梯度（gradient）指出讓損失下降的方向與幅度；對平方誤差，方向由 2 (y_pred − y) 主導，autograd 的結果與手算公式一致。
- 用 autograd 重做梯度下降，更新公式（參數 = 參數 − 學習率 × 梯度）不變，省去手推導數的工作；學習率太大會發散、太小收斂慢。